In [ ]:
import boto3
import json
import requests
import textract
import sagemaker
import time
import logging

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Initialize Amazon Bedrock and AWS Clients
boto3.setup_default_session(region_name="us-east-1")
bedrock_client = boto3.client('bedrock')
s3_client = boto3.client('s3')
athena_client = boto3.client('athena')

# Get SageMaker Execution Role
sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()

# --- Function Implementations ---
def extract_text_from_s3(bucket, key):
    """Extract financial insights from asset PDF reports stored in an S3 bucket."""
    response = s3_client.get_object(Bucket=bucket, Key=key)
    text = textract.process(response['Body'].read()).decode('utf-8')
    return text

def query_athena_for_rebalancing(portfolio_id):
    """Retrieve portfolio rebalancing recommendations from Athena."""
    query = f"""
    SELECT strategy FROM rebalancing_recommendation WHERE portfolio_id = '{portfolio_id}'
    """
    response = athena_client.start_query_execution(
        QueryString=query,
        QueryExecutionContext={'Database': 'investment_db'},
        ResultConfiguration={'OutputLocation': 's3://athena-query-results/'},
    )
    query_execution_id = response['QueryExecutionId']
    return get_athena_query_results(query_execution_id)

def get_athena_query_results(query_execution_id):
    while True:
        response = athena_client.get_query_execution(QueryExecutionId=query_execution_id)
        status = response["QueryExecution"]["Status"]["State"]
        if status in ["SUCCEEDED", "FAILED", "CANCELLED"]:
            break
        time.sleep(2)
    if status == "SUCCEEDED":
        return athena_client.get_query_results(QueryExecutionId=query_execution_id)
    return None

def get_stock_market_insights(asset):
    """Fetch real-time stock market trends related to portfolio assets in the rebalancing strategy."""
    market_data = requests.get(f"https://api.marketdata.com/stocks/{asset}").json()
    return market_data

# --- Create a Guardrail for Financial Compliance ---
def create_guardrail(guardrail_name, policies):
    response = bedrock_client.create_guardrail(
        guardrailName=guardrail_name,
        guardrailDescription="Financial compliance guardrail for AI responses.",
        policies=policies
    )
    return response

financial_guardrail = create_guardrail(
    guardrail_name="FinancialComplianceGuardrail",
    policies=[
        {"name": "SECCompliance", "description": "Ensure investment advice adheres to SEC and FINRA regulations.", "filterType": "REGULATORY"},
        {"name": "BiasMitigation", "description": "Prevent biased financial recommendations.", "filterType": "BIAS"}
    ]
)

# --- Register Functions as Bedrock Tools ---
def register_function_as_tool(function_name, function_description):
    response = bedrock_client.create_tool(
        toolName=function_name,
        toolDescription=function_description
    )
    return response

extract_text_tool = register_function_as_tool(
    "extract_text_from_s3",
    "Extract financial insights from asset PDF reports stored in an S3 bucket."
)

query_athena_tool = register_function_as_tool(
    "query_athena_for_rebalancing",
    "Retrieve portfolio rebalancing recommendations from Athena."
)

market_insights_tool = register_function_as_tool(
    "get_stock_market_insights",
    "Fetch real-time stock market trends related to portfolio assets in the rebalancing strategy."
)

# --- Create Agents and Assign Tools ---
def create_agent(agent_name, instructions, tool_ids):
    response = bedrock_client.create_agent(
        agentName=agent_name,
        agentDescription=f"Agent for {agent_name}",
        foundationModel="anthropic.claude-v2",
        instructions=instructions,
        toolIds=tool_ids
    )
    return response

supervisor_agent = create_agent(
    agent_name="SupervisorAgent",
    instructions="""
    Route user queries to the appropriate sub-agent:
    - Use DataExtractionAgent for financial reports from S3.
    - Use PortfolioAnalysisAgent for investment rebalancing strategies from Athena.
    - Use MarketInsightsAgent for real-time stock market trends related to rebalancing assets.
    """,
    tool_ids=[]
)

data_extraction_agent = create_agent(
    agent_name="DataExtractionAgent",
    instructions="Extract financial insights from asset PDF reports stored in an S3 bucket.",
    tool_ids=[extract_text_tool['toolId']]
)

portfolio_analysis_agent = create_agent(
    agent_name="PortfolioAnalysisAgent",
    instructions="Retrieve portfolio rebalancing recommendations from Athena.",
    tool_ids=[query_athena_tool['toolId']]
)

market_insights_agent = create_agent(
    agent_name="MarketInsightsAgent",
    instructions="Fetch real-time stock market trends related to portfolio assets in the rebalancing strategy.",
    tool_ids=[market_insights_tool['toolId']]
)

# --- Attach Guardrail to the Supervisor Agent ---
def attach_guardrail_to_agent(agent_id, guardrail_id):
    response = bedrock_client.update_agent(
        agentId=agent_id,
        guardrailId=guardrail_id
    )
    return response

attach_guardrail_to_agent(
    agent_id=supervisor_agent['agentId'],
    guardrail_id=financial_guardrail['guardrailId']
)

# --- Invoke the Supervisor Agent ---
def invoke_supervisor_agent(agent_id, user_query):
    response = bedrock_client.invoke_agent(
        agentId=agent_id,
        sessionId="financial-advisor-session",
        inputText=user_query
    )
    return response['outputText']

# --- Validate AI Response Using Guardrails ---
def validate_response_with_guardrail(guardrail_id, response_text):
    validation_response = bedrock_client.validate_response(
        guardrailId=guardrail_id,
        inputText=response_text
    )
    return validation_response['validatedText']

# --- Main Execution ---
def main():
    print("Welcome to the AI Financial Advisor!")
    while True:
        user_input = input("\nAsk a financial question (or type 'exit' to quit): ")
        if user_input.lower() == "exit":
            break
        
        raw_response = invoke_supervisor_agent(supervisor_agent['agentId'], user_input)
        validated_response = validate_response_with_guardrail(financial_guardrail['guardrailId'], raw_response)
        print("\nFinal Validated Response:", validated_response)

if __name__ == "__main__":
    main()
